In [ ]:
pip install -U "bitsandbytes>=0.46.1" transformers accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 42.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


In [2]:
# -*- coding: utf-8 -*-
"""evaluation.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/18i2YWuk0EzItY7xuKYFL2qUkajXrniLV
"""

from google.colab import drive
drive.mount('/content/drive')

import json
import os
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# Path where your trained LoRA adapter is saved in Google Drive
ADAPTER_PATH = "/content/drive/MyDrive/qwen_lora_output"

print("Model:", MODEL_NAME)
print("Adapter Path:", ADAPTER_PATH)

# Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 1. Load Base Model and Tokenizer
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Base Qwen model loaded.")

# 2. Load Fine-Tuned QLoRA Model
qlora_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

qlora_model = PeftModel.from_pretrained(
    qlora_base_model,
    ADAPTER_PATH
)

print("QLoRA model loaded.")

Mounted at /content/drive
Model: Qwen/Qwen2.5-1.5B-Instruct
Adapter Path: /content/drive/MyDrive/qwen_lora_output


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base Qwen model loaded.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

QLoRA model loaded.


In [3]:
import os

print("Directories in /content:")
print([d for d in os.listdir("/content") if os.path.isdir(os.path.join("/content", d))])

if os.path.exists("/content/qwen_lora_output"):
    print("\nContents of /content/qwen_lora_output:")
    print(os.listdir("/content/qwen_lora_output"))

Directories in /content:
['.config', 'drive', 'sample_data']


In [4]:

qlora_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

qlora_model = PeftModel.from_pretrained(
    qlora_base_model,
    ADAPTER_PATH
)

print("QLoRA model loaded.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

QLoRA model loaded.


In [5]:
test_insights = {
    "domain": "sales",

    "metrics": [
        {
            "name": "Total Revenue",
            "value": 1250000,
            "unit": "INR"
        },
        {
            "name": "Profit Margin",
            "value": 19.39,
            "unit": "percent"
        }
    ],

    "trends": [
        {
            "metric": "Revenue",
            "direction": "increasing",
            "change": 12.4,
            "unit": "percent"
        }
    ],

    "comparisons": [
        {
            "dimension": "category",
            "metric": "Revenue",
            "highest": {
                "name": "Electronics",
                "value": 450000
            },
            "lowest": {
                "name": "Furniture",
                "value": 180000
            }
        }
    ],

    "anomalies": []
}

In [6]:
import json


def build_prompt(insights):

    insights_json = json.dumps(
        insights,
        indent=2
    )

    return f"""
You are a business analytics assistant.

Generate a concise business narrative from the
structured analytical findings below.

Rules:
1. Use ONLY the information provided.
2. Do not invent numbers or facts.
3. Do not invent causes.
4. Mention important metrics.
5. Mention important trends.
6. Mention meaningful comparisons.
7. Mention anomalies if present.
8. Keep the narrative concise and professional.

STRUCTURED ANALYTICAL FINDINGS:

{insights_json}

Generate the business narrative now.
"""

In [7]:
def generate_narrative(model, tokenizer, insights):

    prompt = build_prompt(insights)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [8]:
baseline_output = generate_narrative(
    base_model,
    base_tokenizer,
    test_insights
)

print("BASELINE QWEN OUTPUT:")
print(baseline_output)

BASELINE QWEN OUTPUT:
In the domain of sales, the total revenue has shown an increasing trend over the past year, reaching a value of INR 1250000 with a profit margin of 19.39 percent. This indicates that the company is effectively managing its costs while generating substantial income. The highest category in terms of revenue is Electronics, contributing to 450000 units, whereas Furniture stands at 180000 units, indicating a significant disparity between these two categories. No anomalies were observed in the data, suggesting consistent performance across all segments.


In [9]:
qlora_output = generate_narrative(
    qlora_model,
    qlora_tokenizer,
    test_insights
)

print("QLORA OUTPUT:")
print(qlora_output)

QLORA OUTPUT:
The total revenue for the domain has increased by 12.4% to INR 1,250,000 in comparison with the previous period. The profit margin is at 19.39%, indicating that the company's profitability remains strong despite the increase in revenue. Electronics have been the highest contributor to revenue, accounting for 45% of the total, while Furniture contributes only 18%.


In [10]:
print("=" * 80)
print("BASELINE QWEN")
print("=" * 80)
print(baseline_output)

print("\n")

print("=" * 80)
print("QLORA")
print("=" * 80)
print(qlora_output)

BASELINE QWEN
In the domain of sales, the total revenue has shown an increasing trend over the past year, reaching a value of INR 1250000 with a profit margin of 19.39 percent. This indicates that the company is effectively managing its costs while generating substantial income. The highest category in terms of revenue is Electronics, contributing to 450000 units, whereas Furniture stands at 180000 units, indicating a significant disparity between these two categories. No anomalies were observed in the data, suggesting consistent performance across all segments.


QLORA
The total revenue for the domain has increased by 12.4% to INR 1,250,000 in comparison with the previous period. The profit margin is at 19.39%, indicating that the company's profitability remains strong despite the increase in revenue. Electronics have been the highest contributor to revenue, accounting for 45% of the total, while Furniture contributes only 18%.
